# Notebook 01 — Data Ingestion

**BlueStock Fintech Internship — Capstone Project I**

## Objectives
- Load all 10 raw CSV datasets from `data/raw/`
- Print `.shape`, `.dtypes`, `.head()` for each dataset
- Flag data anomalies (nulls, duplicates, negative values, bad dates)
- Explore `fund_master` (unique fund houses, categories, risk grades)
- Validate AMFI codes across `fund_master` ↔ `nav_history`
- Write data quality summary to `reports/data_quality_report.txt`

In [ ]:
from pathlib import Path
import glob
import pandas as pd
import numpy as np
from datetime import datetime

ROOT    = Path('..').resolve()
RAW_DIR = ROOT / 'data' / 'raw'
REP_DIR = ROOT / 'reports'
REP_DIR.mkdir(parents=True, exist_ok=True)

print(f'Root directory : {ROOT}')
print(f'Raw data path  : {RAW_DIR}')
csv_files = sorted(RAW_DIR.glob('*.csv'))
print(f'CSV files found: {len(csv_files)}')

In [ ]:
datasets = {}
for path in csv_files:
    key = path.stem
    df  = pd.read_csv(path, low_memory=False)
    datasets[key] = df
    print(f"{'─'*60}")
    print(f"  {key}  ({df.shape[0]:,} rows × {df.shape[1]} cols)")
    print(df.dtypes.to_string())
    display(df.head(3))

In [ ]:
def flag_anomalies(df: pd.DataFrame, name: str):
    issues = []
    null_counts = df.isnull().sum()
    for col, cnt in null_counts[null_counts>0].items():
        issues.append(f"  [NULL]  '{col}': {cnt:,} nulls ({cnt/len(df)*100:.1f}%)")
    dups = df.duplicated().sum()
    if dups: issues.append(f"  [DUP ]  {dups:,} fully duplicate rows")
    for col in df.select_dtypes(include=[np.number]).columns:
        neg = (df[col] < 0).sum()
        if neg: issues.append(f"  [NEG ]  '{col}': {neg:,} negative values")
    if issues:
        print(f"\nAnomalies in [{name}]:")
        for i in issues: print(i)
    else:
        print(f"[OK] {name} — no anomalies detected.")

for name, df in datasets.items():
    flag_anomalies(df, name)

In [ ]:
# ── Explore fund_master
fm = next((v for k,v in datasets.items() if 'fund_master' in k), None)
if fm is not None:
    for col in ['fund_house','category','sub_category','risk_grade']:
        if col in fm.columns:
            uniq = fm[col].dropna().unique()
            print(f"\n[{col.upper()}] — {len(uniq)} unique values:")
            for v in sorted(uniq): print(f"  {v}")

In [ ]:
# ── AMFI code cross-validation
fm_codes  = set(fm['amfi_code'].dropna().astype(str)) if fm is not None else set()
nav_df    = next((v for k,v in datasets.items() if 'nav_history' in k), None)
nav_codes = set(nav_df['amfi_code'].dropna().astype(str)) if nav_df is not None else set()

print(f"fund_master codes : {len(fm_codes):,}")
print(f"nav_history codes : {len(nav_codes):,}")
print(f"Matched           : {len(fm_codes & nav_codes):,}")
print(f"Missing in nav    : {len(fm_codes - nav_codes):,}")

# Save report
lines = [f"DATA QUALITY REPORT — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}", '='*70]
lines += [f"{k:<35} rows={len(v):>7,} | nulls={v.isnull().sum().sum():>6,} | dups={v.duplicated().sum():>4,}"
           for k,v in datasets.items()]
lines += ['', f'AMFI Matched: {len(fm_codes & nav_codes):,}/{len(fm_codes):,}']
(REP_DIR / 'data_quality_report.txt').write_text('\n'.join(lines), encoding='utf-8')
print('\n✓ Report saved → reports/data_quality_report.txt')